# 1. Classification

**Goal:**
In this notebook, we will perform the first step of the Responsible Data Science project. 
We will:
1. Load and preprocess the `adult.data` dataset.
2. Binarize the `Age` attribute (convert it into two groups).
3. Split the data into **Train**, **Validation**, and **Test** sets.
4. Train a classifier to predict if a person earns `>50K`.
5. Measure the performance of the model.

## Imports

In [16]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from IPython.display import display 
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Set a random state for reproducibility
RANDOM_SEED = 42

## Load the Dataset

The `adult.data` file does not contain headers (column names). We must define them manually based on the dataset documentation. 
We also need to handle spaces in the data (e.g., " White" instead of "White").

In [17]:
import os

# Define column names based on the UCI repository
columns = [
    "Age", "Workclass", "fnlwgt", "Education", "Education-Num",
    "Marital-Status", "Occupation", "Relationship", "Race", "Sex",
    "Capital-Gain", "Capital-Loss", "Hours-per-week", "Country", "Income"
]

# Define the relative path to the data file
# "../" moves up from 'notebooks' to the project root
file_path = '../data/raw/adult.data'

# Check if file exists before loading (good practice)
if not os.path.exists(file_path):
    print(f"Error: File not found at {file_path}")
else:
    # Load the data
    df = pd.read_csv(
        file_path, 
        names=columns, 
        sep=',', 
        skipinitialspace=True, 
        na_values='?'
    )

    # Display the first few rows
    print(f"Dataset shape: {df.shape}")
    display(df.head())

Dataset shape: (32561, 15)


,Age,Workclass,fnlwgt,Education,Education-Num,Marital-Status,Occupation,Relationship,Race,Sex,Capital-Gain,Capital-Loss,Hours-per-week,Country,Income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


## Preprocessing & Imputation Strategy

Instead of simply dropping rows with missing values (which would result in losing ~7% of our data and potentially introducing bias against certain groups), we adopt an **Imputation Strategy**.

* **Missing Values:** We replace missing entries (originally `?`) with the category `"Unknown"`. This allows the model to learn patterns associated with missing information (e.g., perhaps people with specific jobs are less likely to report them) and preserves the dataset size.
* **Age Binarization:** As per the instructions, we convert the continuous `Age` variable into a binary group based on the median:
    * `0`: Young (<= Median)
    * `1`: Senior (> Median)

In [18]:
from IPython.display import display

# 1. Handling Missing Values (Imputation Strategy)

# Check missing values
print("Missing values before imputation:")
# Convert the series to a DataFrame for a nice table display
missing_stats = df.isnull().sum().to_frame(name='Missing Values')
# Show only columns that have missing values (optional, makes it cleaner)
display(missing_stats[missing_stats['Missing Values'] > 0])

# Fill categorical missing values with 'Unknown'
# This prevents losing data (Imputation)
df = df.fillna('Unknown')

print(f"Remaining rows: {df.shape[0]} (We kept everyone!)")
print("-" * 30)

# 2. Binarize Age
median_age = df['Age'].median()
print(f"Median Age: {median_age}")

df['Age'] = (df['Age'] > median_age).astype(int)

print("\nAge distribution (0 = Young, 1 = Senior):")
# Convert value_counts to a DataFrame for a nice table
age_counts = df['Age'].value_counts().to_frame(name='Count')
age_counts.index.name = 'Age Group'
display(age_counts)

Missing values before imputation:


,Missing Values
Workclass,1836
Occupation,1843
Country,583


Remaining rows: 32561 (We kept everyone!)
------------------------------
Median Age: 37.0

Age distribution (0 = Young, 1 = Senior):


,Count
Age Group,
0,16681
1,15880


## Encoding Data

Machine Learning models require numerical input. We cannot give them text directly.

1.  **Target Variable (`Income`):** We map `<=50K` to `0` and `>50K` to `1`.
2.  **Features:** We use **Label Encoding**. Since we filled missing values with `"Unknown"`, this category is simply assigned a unique number just like any other profession or country. This method preserves all the information for the XGBoost classifier without expanding the dataset dimensions unnecessarily.

In [19]:
from sklearn.preprocessing import LabelEncoder

# 1. Encode Target (Income)
df['Income'] = df['Income'].apply(lambda x: 1 if x == '>50K' else 0)

# 2. Separate Features (X) and Target (y)
X = df.drop('Income', axis=1)
y = df['Income']

# 3. Label Encoding
# Standard XGBoost handles numbers better. We encode strings to numbers.
categorical_columns = X.select_dtypes(include=['object']).columns

for col in categorical_columns:
    le = LabelEncoder()
    # "Unknown" will just become a number here, which is perfect
    X[col] = le.fit_transform(X[col].astype(str))

print("Data encoding complete.")
print(f"Features shape: {X.shape}")

Data encoding complete.
Features shape: (32561, 14)


## Split Data

We need three sets:
1.  **Train:** To train the model.
2.  **Validation:** To tune the model (optional, but good practice).
3.  **Test:** To evaluate the final performance.

We will split the data into 60% Train, 20% Validation, and 20% Test.

In [20]:
# First split: 80% for Train+Val, 20% for Test
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED
)

# Second split: Split the 80% (X_temp) into Train (75% of temp) and Val (25% of temp)
# This results in roughly 60% Train, 20% Val, 20% Test of the original total
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=RANDOM_SEED
)

print(f"Training set size: {X_train.shape[0]}")
print(f"Validation set size: {X_val.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")

Training set size: 19536
Validation set size: 6512
Test set size: 6513


## Train the Classifier

### Model Selection Strategy
We selected **XGBoost** (eXtreme Gradient Boosting) for this project. We chose this model based on the trade-off between **performance** and **interpretability**:
1.  **High Performance:** XGBoost captures complex, non-linear relationships in the data better than linear models (like Logistic Regression). This allows us to maximize accuracy.
2.  **Explainability Readiness:** Unlike complex ensembles (like a VotingClassifier), XGBoost is fully compatible with explainability tools like **SHAP** and **Omnixai** (which are required for the "Explainability" step of this project).

In [21]:
# Initialize the XGBoost Classifier
clf = XGBClassifier(
    n_estimators=100, 
    max_depth=6, 
    learning_rate=0.1,
    random_state=RANDOM_SEED, 
    eval_metric='logloss'
)

# Train the model
clf.fit(X_train, y_train)

print("Model training complete (XGBoost).")


display(clf)

Model training complete (XGBoost).


,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'logloss'


### Model Configuration

We explicitly define the **hyperparameters** of our XGBoost model to ensure reproducibility and transparency. Here is what the configuration object represents:

* **`XGBClassifier`**: This is the "Identity Card" of our model. It shows all the settings used to train it.
* **`n_estimators=100`**: The number of "experts" (decision trees) we use. We use 100 trees working together to vote on the final prediction.
* **`max_depth=6`**: The complexity of each tree. A depth of 6 allows the model to learn specific patterns without memorizing the data (overfitting).
* **`learning_rate=0.1`**: The step size at each iteration. A smaller value (like 0.1) makes the training more robust and precise.
* **`random_state=42`**: Ensures that we get exactly the same results every time we run this code.

## Measure Performance

Finally, we evaluate the classifier on the **Test set**.
We look at:
* **Accuracy:** The percentage of correct predictions.
* **Classification Report:** Precision, Recall, and F1-score for each class.

In [22]:
# Predict on the test set (Using X_test, not X_test_scaled)
y_pred = clf.predict(X_test)

# Calculate Accuracy
acc = accuracy_score(y_test, y_pred)
print(f"Test Set Accuracy: {acc:.4f}\n")

# Detailed Report
print("Classification Report:")
print(classification_report(y_test, y_pred))

# Confusion Matrix
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Test Set Accuracy: 0.8781

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.94      0.92      4942
           1       0.79      0.68      0.73      1571

    accuracy                           0.88      6513
   macro avg       0.84      0.81      0.83      6513
weighted avg       0.87      0.88      0.87      6513

Confusion Matrix:
[[4652  290]
 [ 504 1067]]


### Analysis of Classification Results

The XGBoost classifier achieved a **Test Set Accuracy of 87.81%**. This is a strong baseline performance for the Adult dataset. Here is a breakdown of the evaluation metrics:

1.  **Accuracy (Global Performance):**
    * The model correctly predicted the income class for nearly **88%** of the individuals in the test set.

2.  **Confusion Matrix Analysis:**
    * **True Negatives (4652):** The model is very good at identifying people with low income (<=50K).
    * **True Positives (1067):** It successfully identified 1067 high-income earners.
    * **False Negatives (504):** The model missed ~500 rich individuals (classifying them as poor).
    * **False Positives (290):** It incorrectly flagged ~290 poor individuals as rich.
    * *Observation:* The model makes fewer False Positive errors than False Negative errors, which suggests it is slightly conservative in predicting high income.

3.  **Class Imbalance:**
    * The **Precision** for the high-income class (`1`) is **0.79**, while for the low-income class (`0`) it is **0.90**. This difference is expected because the dataset is imbalanced (there are many more people earning <=50K than >50K). Ideally, we want to maintain this high performance while improving fairness in the next steps.

# 2. Fairness

**Goal:**
In this section, we assess the fairness of our classifier regarding two protected attributes: **Sex** and **Age**.
Machine Learning models can unintentionally learn historical biases present in the data (e.g., if women were historically paid less, the model might learn to predict lower income for women).

**Methodology:**
1.  **Metric:** We will use the **Statistical Parity Difference (SPD)**. This metric measures the difference in the probability of receiving a positive outcome (High Income) between the unprivileged group and the privileged group.
    * $SPD = P(Y=1 | Unprivileged) - P(Y=1 | Privileged)$
    * A value of 0 implies perfect fairness.
    * A negative value implies bias against the unprivileged group.
2.  **Mitigation:** We will apply a pre-processing technique called **Reweighing**. Instead of modifying the feature values, this algorithm assigns different **weights** to examples in the training data to ensure that protected groups are represented equally in the positive outcome class.

In [23]:
import sys
import os

# Suppress warnings meant for standard output
# This temporarily hides stderr where aif360 prints its warnings
stderr_backup = sys.stderr
sys.stderr = open(os.devnull, 'w')

# Imports
from aif360.datasets import BinaryLabelDataset
from aif360.metrics import BinaryLabelDatasetMetric
from aif360.algorithms.preprocessing import Reweighing
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier

# Restore stderr so you can see real errors later
sys.stderr = stderr_backup

# We need to reconstruct the training data format for AIF360
train_df = X_train.copy()
train_df['Income'] = y_train.copy()

# Define which values are privileged
privileged_groups = [{'Sex': 1}]
unprivileged_groups = [{'Sex': 0}]

print("Fairness libraries imported and data prepared (Clean output!).")

Fairness libraries imported and data prepared (Clean output!).


## Assess Fairness (Before Mitigation)

First, we transform our Pandas DataFrames into `BinaryLabelDataset` objects, which are required by the `aif360` library.
Then, we calculate the **Statistical Parity Difference** on the original training data to see if there is bias regarding Sex.

In [24]:
# Convert the Pandas DataFrame to an AIF360 BinaryLabelDataset
dataset_orig_train = BinaryLabelDataset(
    favorable_label=1,
    unfavorable_label=0,
    df=train_df,
    label_names=['Income'],
    protected_attribute_names=['Sex', 'Age']
)

# Compute the metric for the original dataset
metric_orig_train = BinaryLabelDatasetMetric(
    dataset_orig_train, 
    unprivileged_groups=unprivileged_groups,
    privileged_groups=privileged_groups
)

spd_orig = metric_orig_train.statistical_parity_difference()

print(f"Original Training Data - Statistical Parity Difference (Sex): {spd_orig:.4f}")

if spd_orig < -0.1:
    print(">> Bias detected: The unprivileged group (Female) is significantly less likely to have high income.")
else:
    print(">> The dataset seems relatively fair.")

Original Training Data - Statistical Parity Difference (Sex): -0.2013
>> Bias detected: The unprivileged group (Female) is significantly less likely to have high income.


### Analysis of Initial Fairness Results

The calculated **Statistical Parity Difference (SPD)** is **-0.2013**. This indicates a significant bias in the original training data. Here is the detailed interpretation:

* **Negative Sign (-):** The negative value confirms that the **unprivileged group** (Females) has a lower probability of receiving the favorable outcome (High Income >50K) compared to the privileged group (Males).
* **Magnitude (0.20):** The absolute difference is roughly **20%**. This means that, in this dataset, men are approximately **20% more likely** to be in the high-income category than women.

**Conclusion:** The dataset reflects historical biases regarding gender and income. Training a model directly on this data without intervention would likely result in a discriminatory classifier. This justifies our decision to apply a mitigation technique (**Reweighing**) in the next step.

## Bias Mitigation: Reweighing

Since we detected bias (SPD is likely negative), we apply the **Reweighing** algorithm.
This method does not change the data values (like Age or Education), but calculates a **weight** for each row.
* Under-represented groups with high income get **higher weights**.
* Over-represented groups with high income get **lower weights**.

This forces the classifier to pay more attention to the under-represented groups during training.

In [25]:
# Initialize the Reweighing algorithm
RW = Reweighing(
    unprivileged_groups=unprivileged_groups,
    privileged_groups=privileged_groups
)

# Fit and transform the training data
dataset_transf_train = RW.fit_transform(dataset_orig_train)

# Verify that the bias is gone in the reweighed dataset
metric_transf_train = BinaryLabelDatasetMetric(
    dataset_transf_train, 
    unprivileged_groups=unprivileged_groups,
    privileged_groups=privileged_groups
)

spd_transf = metric_transf_train.statistical_parity_difference()

print(f"Transformed Training Data - Statistical Parity Difference (Sex): {spd_transf:.4f}")
print(">> The reweighing process has theoretically removed the statistical bias in the dataset.")

Transformed Training Data - Statistical Parity Difference (Sex): -0.0000
>> The reweighing process has theoretically removed the statistical bias in the dataset.


**Analysis of Results:**
The `Reweighing` algorithm successfully reduced the Statistical Parity Difference from **-0.2013** to **-0.0000** on the training data. This indicates that, when the computed weights are applied, the protected group (Female) and the privileged group (Male) have the exact same probability of being assigned the favorable label (High Income). We have successfully created a "fair" training ground for our model.

## Train the Fair Classifier

Now we train a new XGBoost classifier using the weights calculated by the Reweighing algorithm.
**Crucial Step:** We pass the `instance_weights` to the `sample_weight` parameter of XGBoost. This tells the model to respect the fairness correction.

In [26]:
# Extract the weights calculated by AIF360
new_weights = dataset_transf_train.instance_weights

# Initialize the Fair Classifier (Same parameters as before for comparison)
clf_fair = XGBClassifier(
    n_estimators=100, 
    max_depth=6, 
    learning_rate=0.1,
    random_state=RANDOM_SEED, # Use the global random seed
    eval_metric='logloss'
)

# Train using the weights
clf_fair.fit(X_train, y_train, sample_weight=new_weights)

print("Fair Classifier training complete.")
display(clf_fair)

Fair Classifier training complete.


,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'logloss'


## Performance vs. Fairness

We now evaluate both the original classifier and the fair classifier on the **Test Set**.
We verify if we improved fairness (SPD closer to 0) and if we lost any accuracy (Trade-off).

In [27]:
# 1. Predictions on Test Set
y_pred_orig = clf.predict(X_test)      # Original Model (from Step 1)
y_pred_fair = clf_fair.predict(X_test) # Fair Model

# 2. Accuracy
acc_orig = accuracy_score(y_test, y_pred_orig)
acc_fair = accuracy_score(y_test, y_pred_fair)

# 3. Fairness on Test Predictions (Helper function)
def calculate_spd_on_preds(X, y_pred, sensitive_attr='Sex'):
    temp_df = X.copy()
    temp_df['Predicted_Income'] = y_pred
    
    dataset_pred = BinaryLabelDataset(
        favorable_label=1, unfavorable_label=0,
        df=temp_df, label_names=['Predicted_Income'],
        protected_attribute_names=[sensitive_attr]
    )
    
    metric = BinaryLabelDatasetMetric(
        dataset_pred,
        unprivileged_groups=unprivileged_groups,
        privileged_groups=privileged_groups
    )
    return metric.statistical_parity_difference()

spd_pred_orig = calculate_spd_on_preds(X_test, y_pred_orig)
spd_pred_fair = calculate_spd_on_preds(X_test, y_pred_fair)

# 4. Display Results
results = pd.DataFrame({
    'Metric': ['Accuracy', 'Fairness (SPD)'],
    'Original Model': [acc_orig, spd_pred_orig],
    'Fair Model (Reweighed)': [acc_fair, spd_pred_fair]
})

print("Comparison of Original vs Fair Classifier:")
display(results)

Comparison of Original vs Fair Classifier:


,Metric,Original Model,Fair Model (Reweighed)
0,Accuracy,0.878090,0.868417
1,Fairness (SPD),-0.188515,-0.092683


### Interpretation of Results

The table compares the fairness of predictions on the Test Set:

1.  **Original Model (Bias):**
    * **SPD = -0.1885**: This confirms that the original model learned the historical bias. It is roughly **18% less likely** to predict a high income for women than for men.
    * *Verdict:* This creates a significant disparity and is considered unfair.

2.  **Fair Model (Correction):**
    * **SPD = -0.0927**: After training on reweighed data, the disparity dropped to ~9%.
    * *Verdict:* **Success.** In fairness metrics, a Statistical Parity Difference closer to 0 is better. A common industry threshold for "acceptable fairness" is often considered to be within the range of **[-0.1, 0.1]**. Our fair model falls within this acceptable range (-0.09), proving that the mitigation was effective while maintaining high accuracy (86.8%).